In [1]:
from psycopg.rows import dict_row
from psycopg_pool import AsyncConnectionPool
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver

from app.core.config import settings
from dotenv import load_dotenv

load_dotenv()

# 1.创建连接池
checkpoint_pool = AsyncConnectionPool(
    conninfo=settings.db.checkpoint_url,
    min_size=3,  # 池最小连接数
    max_size=10,  # 池最大连接数
    kwargs={
        "autocommit": True,  # 自动提交事务
        "prepare_threshold": 0,  # 不做预准备sql
        "row_factory": dict_row,  # 查询结果以dict返回
    },
    open=False,  # 不自动创建连接
)
# 初始化连接
await checkpoint_pool.open()
# 等待连接池就绪
await checkpoint_pool.wait()

# 初始化checkpointer
checkpointer = AsyncPostgresSaver(checkpoint_pool)
# 自动建表（与checkpointer存储有关的表）
await checkpointer.setup()

In [20]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage

# 1.加载环境变量
from dotenv import load_dotenv
load_dotenv()

# 2.初始化模型
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

# 3.定义工具


# 4.创建Agent，绑定模型和工具，Checkpointer
agent = create_agent(
    model = model,
    tools=[],
    checkpointer=checkpointer
)

In [32]:
# 5.调用Agent，要能查询记忆，必须携带thread_id
config = {"configurable": {"thread_id": "thread_01"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage("12个苹果分给4个人，每人分几个？")]},
    config=config,
)
print(response['messages'][-1].content)

这是一个简单的除法问题。

把12个苹果平均分给4个人，就是计算：
12 ÷ 4 = 3

所以，**每人分3个苹果**。


In [33]:
response = await agent.ainvoke(
    {"messages": [HumanMessage("如果是3个人呢？")]},
    config=config,
)
print(response['messages'][-1].content)

如果是**平均分**给3个人，同样是除法：

12 ÷ 3 = 4

所以，**每人分4个苹果**。


In [34]:
# 查询会话历史
state_snapshot = await agent.aget_state(config)
print(state_snapshot)

StateSnapshot(values={'messages': [HumanMessage(content='12个苹果分给4个人，每人分几个？', additional_kwargs={}, response_metadata={}, id='8d267498-0160-4e1c-8765-d2c02d4a37b5'), AIMessage(content='这是一个简单的除法问题。\n\n把12个苹果平均分给4个人，就是计算：\n12 ÷ 4 = 3\n\n所以，**每人分3个苹果**。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 16, 'total_tokens': 52, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 16}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': 'a00ee53b-9a5d-49bf-8cdb-7b3e47a053db', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ffe61-d206-7d23-8d09-3d7b6b49e6a1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 36, 'total_tokens': 52, 'input_token_details': {'cache_re

In [ ]:
{"role": "", "content": ""}
# SystemMessage -> system
# HumanMessage -> user
# AIMessage -> assistant
# ToolMessage -> tool


In [40]:
from langchain.messages import HumanMessage, AIMessage

messages = []

for message in state_snapshot.values['messages']:
    # 判断类型，决定role
    if isinstance(message, HumanMessage):
        role = "user"
    elif isinstance(message, AIMessage):
        role = "assistant"
    else:
        continue

    messages.append({"role": role, "content": message.text})

print(messages)

[{'role': 'user', 'content': 'hi'}, {'role': 'assistant', 'content': '你好！我是安心保智能保险顾问，很高兴为你服务！😊\n\n我可以帮你解答各种保险相关的问题，比如：\n- 保险产品咨询（医疗险、重疾险、意外险、寿险等）\n- 保险方案规划建议\n- 理赔流程指导\n- 保险条款解读\n\n请问有什么可以帮到你？或者你想了解哪方面的保险内容呢？'}]


In [26]:
await checkpointer.adelete_thread("7d63e96f-b8c2-4781-b9cc-faf7eefa3ea5")

In [27]:
state_snapshot = await agent.aget_state(config)
print(state_snapshot)

StateSnapshot(values={}, next=(), config={'recursion_limit': 9999, 'metadata': {'ls_integration': 'langchain_create_agent'}, 'configurable': {'thread_id': 'thread_01'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())


In [39]:
config = {"configurable": {"thread_id": "de55b937-1265-4122-961a-c22accc07ab3"}}
state_snapshot = await agent.aget_state(config)
print(state_snapshot)

StateSnapshot(values={'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='9fad887d-1dce-4fae-9536-a893f5597009'), AIMessage(content=[{'type': 'text', 'text': '你好！我是安心保智能保险顾问，很高兴为你服务！😊\n\n我可以帮你解答各种保险相关的问题，比如：\n- 保险产品咨询（医疗险、重疾险、意外险、寿险等）\n- 保险方案规划建议\n- 理赔流程指导\n- 保险条款解读\n\n请问有什么可以帮到你？或者你想了解哪方面的保险内容呢？', 'index': 0}], additional_kwargs={}, response_metadata={'model_provider': 'deepseek', 'finish_reason': 'stop', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'output_version': 'v1'}, id='lc_run--019ffe55-81e3-7530-9f02-973b7acd7dcc', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 80, 'total_tokens': 128, 'input_token_details': {'cache_read': 0}})]}, next=(), config={'configurable': {'thread_id': 'de55b937-1265-4122-961a-c22accc07ab3', 'checkpoint_ns': '', 'checkpoint_id': '1f197910-03bd-694c-8001-887821f509b5'}}, metadata={'step': 1, 'source': 'loop', 'parents': {}, 'l